# Лабораторная работа 4. Динамический контент — вариант 2

**Источник (динамика):** `quotes.toscrape.com/js/` и `quotes.toscrape.com/js-delayed/` — клиентский рендеринг, JSON в `<script>`, задержка отрисовки.
**Контрольный статический источник:** `quotes.toscrape.com/`.

На каждом источнике 10 страниц по 10 цитат. Поэтому набор ≥ 200 записей собирается с **двух источников** (`/js/` + `/js-delayed/`) с ведением журнала. Каждый переход по кнопке **Next** — это цикл взаимодействия, всего их ~20.

**Правила, соблюдённые в коде.**
- Нет `time.sleep` для ожидания данных: используются `wait_for`, `expect`, `wait_for_url` и `wait_for_function`. `sleep` применяется только как пауза вежливости.
- Нет `while True`: все циклы ограничены.
- Нет `except: pass`.
- Нет структурного XPath и сгенерированных классов.
- Браузер закрывается в `finally`.
- Логин выполняется один раз, затем сессия переиспользуется через `storage_state`.

## 0. Подготовка

Каждый этап с браузером запускается через `with_browser(...)`: `async with async_playwright()` + `try/finally browser.close()`. Поэтому при ошибке в ячейке процесс Chromium не остаётся висеть в памяти.

In [5]:
# Установка
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "playwright", "psutil", "pandas", "beautifulsoup4", "requests"], check=True)
cmd = [sys.executable, "-m", "playwright", "install", "chromium"]
if sys.platform.startswith("linux"):
    cmd.append("--with-deps")
r = subprocess.run(cmd, capture_output=True, text=True)
print("chromium установлен" if r.returncode == 0 else r.stdout[-1500:] + r.stderr[-1500:])

chromium установлен


In [6]:
import asyncio, json, re, time, os, sys, threading, statistics
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
import pandas as pd
import psutil
from playwright.async_api import async_playwright, expect, TimeoutError as PWTimeout

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)

BASE       = "https://quotes.toscrape.com"
SOURCES    = {"js": f"{BASE}/js/", "js-delayed": f"{BASE}/js-delayed/"}
STATIC_URL = f"{BASE}/"
LOGIN_URL  = f"{BASE}/login"
UA         = "course-crawler/1.0 (educational use; lab4 variant 2)"

ART        = Path("artifacts"); ART.mkdir(exist_ok=True)
STATE_FILE = Path("auth_state.json")

# Предохранители и параметры сбора
TARGET_TOTAL     = 200                  # требование: >= 200 записей
PER_SOURCE_LIMIT = 100                  # предохранитель: лимит записей на источник
MAX_PAGES        = 15                   # предохранитель: максимум итераций на источник
STALL_LIMIT      = 2                    # предохранитель: N итераций подряд без прироста
POLITE_PAUSE     = 0.5                  # пауза вежливости между итерациями
ITEM_TIMEOUT     = {"js": 10_000, "js-delayed": 25_000}   # мс: у /js-delayed/ отрисовка с задержкой

QUOTE   = ".quote"
DATA_RE = re.compile(r"var\s+data\s*=\s*(\[.*?\]);", re.S)

# HTTP-сессия для дешёвой альтернативы: повторы с экспоненциальной паузой
SESSION = requests.Session()
SESSION.headers["User-Agent"] = UA
SESSION.mount("https://", HTTPAdapter(max_retries=Retry(
    total=3, backoff_factor=1.0, status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=("GET",))))

def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def page_number(url):
    m = re.search(r"/page/(\d+)/", url)
    return int(m.group(1)) if m else 1

print("Конфигурация загружена. Источники:", SOURCES)

Конфигурация загружена. Источники: {'js': 'https://quotes.toscrape.com/js/', 'js-delayed': 'https://quotes.toscrape.com/js-delayed/'}


### Общие утилиты

- `with_browser` открывает и гарантированно закрывает браузер.
- `new_context` создаёт контекст с `user_agent`, `viewport`, `locale` и таймаутами по умолчанию. Опционально он блокирует медиа и подхватывает `storage_state`.
- `NetStats` считает сетевые запросы, трафик и XHR через события контекста.
- `PeakMem` фоновым потоком замеряет пиковую память. В сумму входят процесс Python и все дочерние процессы, то есть драйвер Playwright и Chromium.
- `save_artifacts` сохраняет скриншот и HTML при сбое.

In [7]:
BLOCKED_TYPES = {"image", "media", "font"}

async def _block_router(route):
    if route.request.resource_type in BLOCKED_TYPES:
        await route.abort()
    else:
        await route.continue_()

async def _with_browser_impl(job, headless):
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=headless)
        try:
            return await job(browser)
        finally:
            await browser.close()

def _run_in_own_loop(job, headless):
    """Windows: свой ProactorEventLoop в отдельном потоке (поддерживает подпроцессы)."""
    loop = asyncio.ProactorEventLoop()
    try:
        return loop.run_until_complete(_with_browser_impl(job, headless))
    finally:
        loop.close()

async def with_browser(job, *, headless=True):
    """Запускает job(browser) и гарантированно закрывает браузер и Playwright."""
    if sys.platform == "win32":
        return await asyncio.get_running_loop().run_in_executor(None, _run_in_own_loop, job, headless)
    return await _with_browser_impl(job, headless)

async def new_context(browser, *, block_media=False, storage_state=None, js=True):
    state = str(storage_state) if storage_state and Path(storage_state).exists() else None
    ctx = await browser.new_context(
        user_agent=UA,
        viewport={"width": 1280, "height": 900},
        locale="ru-RU",
        java_script_enabled=js,
        storage_state=state,
    )
    ctx.set_default_timeout(15_000)               # действия и ожидания
    ctx.set_default_navigation_timeout(30_000)    # переходы
    if block_media:
        await ctx.route("**/*", _block_router)
    return ctx

class NetStats:
    """Счётчик сети по событиям контекста: запросы, типы, трафик, XHR/fetch, отменённые."""
    def __init__(self):
        self.requests, self.failed, self.bytes = 0, 0, 0
        self.types, self.xhr, self.size_errors = Counter(), [], []
        self._pending = []
    def attach(self, ctx):
        ctx.on("request", self._on_request)
        ctx.on("requestfailed", self._on_failed)
        ctx.on("requestfinished", lambda req: self._pending.append(asyncio.ensure_future(self._size(req))))
        return self
    def _on_request(self, req):
        self.requests += 1
        self.types[req.resource_type] += 1
        if req.resource_type in ("xhr", "fetch"):
            self.xhr.append(f"{req.method} {req.url}")
    def _on_failed(self, req):
        self.failed += 1                           # сюда попадают и запросы, отменённые route.abort()
    async def _size(self, req):
        try:
            s = await req.sizes()
            self.bytes += max(0, s["responseBodySize"]) + max(0, s["responseHeadersSize"])
        except Exception as e:                     # не глушим: фиксируем, по какому URL размер не получен
            self.size_errors.append(f"{req.url}: {type(e).__name__}")
    async def flush(self):
        if self._pending:
            await asyncio.gather(*self._pending, return_exceptions=True)
            self._pending.clear()
    def summary(self):
        return {"запросов": self.requests, "отменено/ошибок": self.failed,
                "трафик, МБ": round(self.bytes / 1e6, 3), "XHR/fetch": len(self.xhr)}

class PeakMem:
    """Пиковая RSS процесса Python + всех дочерних (драйвер Playwright, Chromium)."""
    def __init__(self, interval=0.1):
        self.interval, self.peak, self._stop = interval, 0, threading.Event()
    def _total(self):
        root = psutil.Process()
        total = 0
        for p in [root] + root.children(recursive=True):
            try:
                total += p.memory_info().rss
            except psutil.Error:                   # процесс завершился между перечислением и замером
                continue
        return total
    def _run(self):
        while not self._stop.is_set():
            self.peak = max(self.peak, self._total())
            self._stop.wait(self.interval)
    def __enter__(self):
        self._t = threading.Thread(target=self._run, daemon=True); self._t.start(); return self
    def __exit__(self, *exc):
        self._stop.set(); self._t.join()
    @property
    def peak_mb(self):
        return round(self.peak / 2**20, 1)

async def save_artifacts(page, tag):
    """Скриншот + HTML текущего состояния страницы в artifacts/."""
    base = ART / f"{tag}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    await page.screenshot(path=f"{base}.png", full_page=True)
    Path(f"{base}.html").write_text(await page.content(), encoding="utf-8")
    return base

# Извлечение всех цитат страницы ОДНИМ вызовом JS в странице
EXTRACT_JS = """
els => els.map((el, i) => ({
    position: i + 1,
    text:   el.querySelector('.text')?.innerText?.trim()   ?? null,
    author: el.querySelector('.author')?.innerText?.trim() ?? null,
    tags:   Array.from(el.querySelectorAll('.tag')).map(t => t.innerText.trim())
}))
"""
print("Утилиты готовы")

Утилиты готовы


## 1. Диагностика источника

Порядок проверки:
1. Браузер рендерит страницу. Фиксируем, сколько появилось `.quote`, сколько прошло времени до первой цитаты и какие запросы ушли в сеть, отдельно XHR/fetch.
2. Тот же адрес загружается с **отключённым JS**: смотрим, что остаётся доступным.
3. `requests` получает ответ сервера: размер HTML и число `.quote`. Затем ищем текст первой отрендеренной цитаты в исходном HTML и определяем, где именно он лежит.
4. Контрольный опыт на статической версии.

In [8]:
async def diagnose_browser(browser):
    out = {}
    for name, url in SOURCES.items():
        # --- JS включён
        ctx = await new_context(browser)
        net = NetStats().attach(ctx)
        try:
            page = await ctx.new_page()
            t0 = time.perf_counter()
            await page.goto(url, wait_until="domcontentloaded")
            t_dom = time.perf_counter() - t0
            await page.locator(QUOTE).first.wait_for(state="visible", timeout=ITEM_TIMEOUT[name])
            t_render = time.perf_counter() - t0
            n_quotes = await page.locator(QUOTE).count()
            first_text = await page.locator(QUOTE).first.locator(".text").inner_text()
            html_rendered = await page.content()
            await net.flush()
        finally:
            await ctx.close()
        # --- JS выключен
        ctx_off = await new_context(browser, js=False)
        try:
            p = await ctx_off.new_page()
            await p.goto(url, wait_until="load")
            n_off      = await p.locator(QUOTE).count()
            next_off   = await p.get_by_role("link", name="Next").count() > 0
            login_off  = await p.get_by_role("link", name="Login").count() > 0
            body_off   = re.sub(r"\s+", " ", await p.locator("body").inner_text())[:160]
            await p.screenshot(path=str(ART / f"diag_{name}_js_off.png"), full_page=True)
        finally:
            await ctx_off.close()
        out[name] = dict(t_dom=t_dom, t_render=t_render, n_quotes=n_quotes, first_text=first_text,
                         rendered_bytes=len(html_rendered), net=net, n_off=n_off,
                         next_off=next_off, login_off=login_off, body_off=body_off)
    return out

diag_br = await with_browser(diagnose_browser)
for k, v in diag_br.items():
    print(f"[{k}] .quote после JS: {v['n_quotes']} | DOMContentLoaded {v['t_dom']:.2f} c | "
          f"первая цитата через {v['t_render']:.2f} c | запросов {v['net'].requests}, XHR/fetch {len(v['net'].xhr)}")
    print(f"      типы ресурсов: {dict(v['net'].types)}")
    print(f"      JS выключен: .quote = {v['n_off']}, ссылка Next = {v['next_off']}, Login = {v['login_off']}")

[js] .quote после JS: 10 | DOMContentLoaded 1.97 c | первая цитата через 2.04 c | запросов 7, XHR/fetch 0
      типы ресурсов: {'document': 1, 'stylesheet': 3, 'script': 1, 'font': 2}
      JS выключен: .quote = 0, ссылка Next = True, Login = True
[js-delayed] .quote после JS: 10 | DOMContentLoaded 1.61 c | первая цитата через 11.71 c | запросов 7, XHR/fetch 0
      типы ресурсов: {'document': 1, 'stylesheet': 3, 'script': 1, 'font': 2}
      JS выключен: .quote = 0, ссылка Next = True, Login = True


In [9]:
def diagnose_server(name, url, target_text):
    r = SESSION.get(url, timeout=20)
    r.raise_for_status()
    html = r.text
    soup = BeautifulSoup(html, "html.parser")
    probe = target_text.strip("“”\"")[:40]                  # фрагмент текста первой отрисованной цитаты
    in_html = probe in html
    scripts = [s.get_text() for s in soup.find_all("script")]
    in_script = any(probe in s for s in scripts)
    m = DATA_RE.search(html)
    n_json = len(json.loads(m.group(1))) if m else 0
    timer = re.search(r"setTimeout\s*\((.{0,160})", html, re.S)
    where = "не найден"
    if in_script and m:
        where = f"<script>: var data = [...] ({n_json} объектов)"
    elif in_script:
        where = "<script> (иной формат)"
    elif in_html:
        where = "в разметке вне <script>"
    return dict(
        status=r.status_code, html_bytes=len(r.content), quotes_server=len(soup.select(QUOTE)),
        probe=probe, in_html=in_html, where=where, n_json=n_json,
        json_ld="application/ld+json" in html, next_data="__NEXT_DATA__" in html,
        timer=re.sub(r"\s+", " ", timer.group(0))[:120] if timer else "нет",
    )

diag_srv = {name: diagnose_server(name, url, diag_br[name]["first_text"]) for name, url in SOURCES.items()}
for k, v in diag_srv.items():
    print(f"[{k}] HTTP {v['status']}, {v['html_bytes']} байт, .quote = {v['quotes_server']}, "
          f"текст «{v['probe'][:30]}…» найден: {v['in_html']} → {v['where']}")
    print(f"      JSON-LD: {v['json_ld']}, __NEXT_DATA__: {v['next_data']}, таймер: {v['timer']}")

# Контрольный статический источник
st = SESSION.get(STATIC_URL, timeout=20)
st_quotes = len(BeautifulSoup(st.text, "html.parser").select(QUOTE))
print(f"\n[static] {STATIC_URL}: {len(st.content)} байт, .quote = {st_quotes}")

[js] HTTP 200, 5808 байт, .quote = 0, текст «The world as we have created i…» найден: True → <script>: var data = [...] (10 объектов)
      JSON-LD: False, __NEXT_DATA__: False, таймер: нет
[js-delayed] HTTP 200, 6014 байт, .quote = 0, текст «The world as we have created i…» найден: True → <script>: var data = [...] (10 объектов)
      JSON-LD: False, __NEXT_DATA__: False, таймер: setTimeout(function() { var data = [ { "tags": [ "change", "deep-thoughts", "thinking", "world" 

[static] https://quotes.toscrape.com/: 11064 байт, .quote = 10


In [10]:
# Протокол диагностики источника (таблица для отчёта)
def _xhr_list(net):
    return "; ".join(net.xhr) if net.xhr else "нет XHR/fetch-запросов"

protocol = pd.DataFrame({
    name: {
        "Ответ сервера (requests): размер HTML, байт":           diag_srv[name]["html_bytes"],
        "Ответ сервера: найдено .quote":                         diag_srv[name]["quotes_server"],
        "Целевой текст в исходном HTML":                         "найден" if diag_srv[name]["in_html"] else "не найден",
        "Где именно найден":                                     diag_srv[name]["where"],
        "JSON-LD / __NEXT_DATA__":                               f"{diag_srv[name]['json_ld']} / {diag_srv[name]['next_data']}",
        "Таймер отрисовки в скрипте":                            diag_srv[name]["timer"],
        "Браузер: найдено .quote":                               diag_br[name]["n_quotes"],
        "Браузер: размер DOM после JS, байт":                    diag_br[name]["rendered_bytes"],
        "Браузер: время до первой цитаты, с":                    round(diag_br[name]["t_render"], 2),
        "Network: всего запросов":                               diag_br[name]["net"].requests,
        "Network: Fetch/XHR (метод URL)":                        _xhr_list(diag_br[name]["net"]),
        "Network: типы ресурсов":                                dict(diag_br[name]["net"].types),
        "JS отключён: найдено .quote":                           diag_br[name]["n_off"],
        "JS отключён: доступна ссылка Next / Login":             f"{diag_br[name]['next_off']} / {diag_br[name]['login_off']}",
        "JS отключён: начало видимого текста":                   diag_br[name]["body_off"],
        "Контроль (статическая версия): найдено .quote":         st_quotes,
        "Вывод":                                                 "",
    } for name in SOURCES
})
protocol.to_csv("diagnostics_protocol.csv", encoding="utf-8-sig")
display(protocol)
print("Скриншоты страниц с выключенным JS:", sorted(p.name for p in ART.glob("diag_*")))

,js,js-delayed
"Ответ сервера (requests): размер HTML, байт",5808,6014
Ответ сервера: найдено .quote,0,0
Целевой текст в исходном HTML,найден,найден
Где именно найден,<script>: var data = [...] (10 объектов),<script>: var data = [...] (10 объектов)
JSON-LD / __NEXT_DATA__,False / False,False / False
Таймер отрисовки в скрипте,нет,"setTimeout(function() { var data = [ { ""tags"": [ ""change"", ""deep-t..."
Браузер: найдено .quote,10,10
"Браузер: размер DOM после JS, байт",8940,9146
"Браузер: время до первой цитаты, с",2.04,11.71
Network: всего запросов,7,7


Скриншоты страниц с выключенным JS: ['diag_js-delayed_js_off.png', 'diag_js_js_off.png']


## 2. Каркас Playwright: первые 10 записей, массовое vs поэлементное извлечение

На первой странице `/js/` получаем первые 10 записей двумя способами:
- одним вызовом `locator.evaluate_all(EXTRACT_JS)`;
- поэлементно через `inner_text` у каждого локатора.

Каждый способ выполняется 5 раз, сравниваются медианы. Дополнительно проверяем, есть ли в отрендеренном DOM `data-*`- и `itemprop`-атрибуты: это нужно для выбора локаторов (раздел 8).

In [11]:
async def frame_demo(browser):
    ctx = await new_context(browser)
    try:
        page = await ctx.new_page()
        await page.goto(SOURCES["js"], wait_until="domcontentloaded")
        quotes = page.locator(QUOTE)
        await expect(quotes.first).to_be_visible()                 # авто-ожидание отрисовки

        bulk_t, item_t = [], []
        for _ in range(5):
            t0 = time.perf_counter()
            bulk = await quotes.evaluate_all(EXTRACT_JS)
            bulk_t.append(time.perf_counter() - t0)

            t0 = time.perf_counter()
            per_item = []
            for q in await quotes.all():
                per_item.append({"text":   await q.locator(".text").inner_text(),
                                 "author": await q.locator(".author").inner_text(),
                                 "tags":   await q.locator(".tag").all_inner_texts()})
            item_t.append(time.perf_counter() - t0)

        attrs = await page.evaluate("""() => ({
            data_attr: [...document.querySelectorAll('*')].filter(e =>
                         [...e.attributes].some(a => a.name.startsWith('data-'))).length,
            itemprop:  document.querySelectorAll('[itemprop]').length,
            testid:    document.querySelectorAll('[data-testid]').length })""")
        return bulk, per_item, bulk_t, item_t, attrs
    finally:
        await ctx.close()

first10, first10_slow, bulk_t, item_t, dom_attrs = await with_browser(frame_demo)
bulk_ms, item_ms = statistics.median(bulk_t) * 1000, statistics.median(item_t) * 1000
extract_cmp = {"evaluate_all, мс (медиана)": round(bulk_ms, 1),
               "поэлементно, мс (медиана)": round(item_ms, 1),
               "выигрыш, раз": round(item_ms / max(bulk_ms, 1e-6), 1)}
print("Получено записей:", len(first10), "| совпадают со способом поэлементно:",
      [(r["text"], r["author"]) for r in first10] == [(r["text"], r["author"]) for r in first10_slow])
print("Сравнение извлечения:", extract_cmp)
print("Атрибуты в отрендеренном DOM /js/:", dom_attrs)
display(pd.DataFrame(first10).head(10))

Получено записей: 10 | совпадают со способом поэлементно: True
Сравнение извлечения: {'evaluate_all, мс (медиана)': 6.5, 'поэлементно, мс (медиана)': 136.6, 'выигрыш, раз': 20.9}
Атрибуты в отрендеренном DOM /js/: {'data_attr': 0, 'itemprop': 0, 'testid': 0}


,position,text,author,tags
0,1,“The world as we have created it is a process of our thinking. It ...,Albert Einstein,"[change, deep-thoughts, thinking, world]"
1,2,"“It is our choices, Harry, that show what we truly are, far more t...",J.K. Rowling,"[abilities, choices]"
2,3,“There are only two ways to live your life. One is as though nothi...,Albert Einstein,"[inspirational, life, live, miracle, miracles]"
3,4,"“The person, be it gentleman or lady, who has not pleasure in a go...",Jane Austen,"[aliteracy, books, classic, humor]"
4,5,"“Imperfection is beauty, madness is genius and it's better to be a...",Marilyn Monroe,"[be-yourself, inspirational]"
5,6,“Try not to become a man of success. Rather become a man of value.”,Albert Einstein,"[adulthood, success, value]"
6,7,“It is better to be hated for what you are than to be loved for wh...,André Gide,"[life, love]"
7,8,"“I have not failed. I've just found 10,000 ways that won't work.”",Thomas A. Edison,"[edison, failure, inspirational, paraphrased]"
8,9,“A woman is like a tea bag; you never know how strong it is until ...,Eleanor Roosevelt,[misattributed-eleanor-roosevelt]
9,10,"“A day without sunshine is like, you know, night.”",Steve Martin,"[humor, obvious, simile]"


## 3. Сценарий взаимодействия

**Авторизация:** вход выполняется один раз, состояние сохраняется в `auth_state.json`. При повторных запусках файл подхватывается, и повторного логина нет. Все контексты сбора создаются из `storage_state`.

**Пагинация:** выполняется кликом по `get_by_role("link", name="Next")`. Это один цикл взаимодействия.

**Ожидания (без `sleep`):**
- `wait_for_url("**/page/N/")` ждёт завершения перехода;
- `locator(".quote").first.wait_for(state="visible")` ждёт отрисовки. Для `/js-delayed/` таймаут 25 с.

**Предохранители:**
1. не больше `MAX_PAGES` итераций;
2. остановка после `STALL_LIMIT` итераций подряд без новых записей;
3. лимит `PER_SOURCE_LIMIT` записей.

Дополнительно цикл останавливается, если ссылки Next больше нет.

**Сбой страницы:** при таймауте сохраняются скриншот и HTML, в журнал пишется строка со `status=timeout`, затем цикл переходит к следующей странице.

In [12]:
async def ensure_auth_state(browser):
    """Логин один раз; при наличии auth_state.json повторный вход не выполняется."""
    if STATE_FILE.exists():
        return "переиспользован существующий auth_state.json"
    ctx = await new_context(browser)
    try:
        page = await ctx.new_page()
        await page.goto(LOGIN_URL)
        await page.get_by_label("Username").fill("student")
        await page.get_by_label("Password").fill("student")
        await page.get_by_role("button", name="Login").click()
        await expect(page.get_by_role("link", name="Logout")).to_be_visible()
        await ctx.storage_state(path=str(STATE_FILE))
        return "выполнен вход, сохранён auth_state.json"
    finally:
        await ctx.close()

async def collect_source(ctx, source, *, max_pages=MAX_PAGES, max_items=PER_SOURCE_LIMIT,
                         stall_limit=STALL_LIMIT, item_timeout=None, pause=POLITE_PAUSE,
                         wait_until="domcontentloaded"):
    """Сбор одного источника кликами по Next. Возвращает (записи, журнал, причина остановки)."""
    item_timeout = item_timeout or ITEM_TIMEOUT[source]
    page = await ctx.new_page()
    records, log, seen = [], [], set()
    stalls, stop_reason = 0, "max_pages"
    try:
        await page.goto(SOURCES[source], wait_until=wait_until)
        for it in range(1, max_pages + 1):                         # предохранитель 1
            t0, url, page_no, status = time.perf_counter(), page.url, page_number(page.url), "ok"
            try:
                await page.locator(QUOTE).first.wait_for(state="visible", timeout=item_timeout)
                batch = await page.locator(QUOTE).evaluate_all(EXTRACT_JS)
            except PWTimeout as e:
                status, batch = "timeout", []
                base = await save_artifacts(page, f"{source}_p{page_no}")
                print(f"  ⚠ [{source}] стр.{page_no}: {str(e).splitlines()[0][:90]} → артефакты {base.name}.*")

            new = 0
            for rec in batch:
                key = (source, rec["text"])
                if rec["text"] and key in seen:
                    continue
                if rec["text"]:
                    seen.add(key)
                rec.update(source=source, page=page_no, page_url=url, collected_at=now_iso())
                records.append(rec); new += 1

            log.append({"source": source, "iteration": it, "url": url, "elements": len(batch),
                        "increment": new, "total": len(records),
                        "duration_s": round(time.perf_counter() - t0, 3),
                        "timestamp": now_iso(), "status": status})

            if new == 0:                                           # предохранитель 2
                stalls += 1
                if stalls >= stall_limit:
                    stop_reason = "stall"; break
            else:
                stalls = 0
            if len(records) >= max_items:                          # предохранитель 3
                stop_reason = "max_items"; break
            nxt = page.get_by_role("link", name="Next")
            if await nxt.count() == 0:
                stop_reason = "last_page"; break
            await asyncio.sleep(pause)                             # пауза вежливости между итерациями
            await nxt.click()                                      # цикл взаимодействия
            await page.wait_for_url(f"**/page/{page_no + 1}/", wait_until=wait_until)
    finally:
        await page.close()
    return records[:max_items], log, stop_reason

In [13]:
async def main_browser_run(browser):
    auth_msg = await ensure_auth_state(browser)
    ctx = await new_context(browser, block_media=True, storage_state=STATE_FILE)
    net = NetStats().attach(ctx)
    all_recs, all_log, reasons = [], [], {}
    try:
        chk = await ctx.new_page()                                  # контекст авторизован без нового входа
        await chk.goto(STATIC_URL)                                 # шапка /js/ статус входа не показывает
        logged = await chk.get_by_role("link", name="Logout").count() > 0
        await chk.close()
        for src in SOURCES:
            recs, log, reason = await collect_source(ctx, src)
            all_recs += recs; all_log += log; reasons[src] = reason
            print(f"[{src}] записей: {len(recs)}, итераций: {len(log)}, остановка: {reason}")
        await net.flush()
    finally:
        await ctx.close()
    return all_recs, all_log, reasons, net, auth_msg, logged

with PeakMem() as mem_browser:
    t0 = time.perf_counter()
    recs_b, log_b, reasons_b, net_b, auth_msg, logged = await with_browser(main_browser_run)
    browser_time = time.perf_counter() - t0

print("Авторизация:", auth_msg, "| в контексте сбора виден Logout:", logged)

df_browser = pd.DataFrame(recs_b)
df_browser["tags"] = df_browser["tags"].apply(lambda v: "; ".join(v) if isinstance(v, list) else v)
df_browser = df_browser[["source", "page", "position", "text", "author", "tags", "page_url", "collected_at"]]
df_log = pd.DataFrame(log_b)
df_log.insert(0, "cycle", range(1, len(df_log) + 1))

df_browser.to_csv("data_browser.csv", index=False, encoding="utf-8-sig")
df_log.to_csv("collection_log.csv", index=False, encoding="utf-8-sig")

clicks = int((df_log.groupby("source")["iteration"].max() - 1).sum())
print(f"\nИтого: {len(df_browser)} записей за {browser_time:.1f} c | циклов (страниц): {len(df_log)}, "
      f"из них переходов кликом Next: {clicks}")
print("Сеть:", net_b.summary(), "| пиковая память:", mem_browser.peak_mb, "МБ")
assert len(df_browser) >= TARGET_TOTAL, f"Собрано меньше {TARGET_TOTAL}"
display(df_log)

[js] записей: 100, итераций: 10, остановка: max_items
[js-delayed] записей: 100, итераций: 10, остановка: max_items
Авторизация: переиспользован существующий auth_state.json | в контексте сбора виден Logout: True

Итого: 200 записей за 133.0 c | циклов (страниц): 20, из них переходов кликом Next: 18
Сеть: {'запросов': 146, 'отменено/ошибок': 42, 'трафик, МБ': 1.189, 'XHR/fetch': 0} | пиковая память: 560.2 МБ


,cycle,source,iteration,url,elements,increment,total,duration_s,timestamp,status
0,1,js,1,https://quotes.toscrape.com/js/,10,10,10,0.121,2026-11-17T11:29:29+00:00,ok
1,2,js,2,https://quotes.toscrape.com/js/page/2/,10,10,20,0.087,2026-11-17T11:29:31+00:00,ok
2,3,js,3,https://quotes.toscrape.com/js/page/3/,10,10,30,0.087,2026-11-17T11:29:33+00:00,ok
3,4,js,4,https://quotes.toscrape.com/js/page/4/,10,10,40,0.083,2026-11-17T11:29:34+00:00,ok
4,5,js,5,https://quotes.toscrape.com/js/page/5/,10,10,50,0.084,2026-11-17T11:29:36+00:00,ok
5,6,js,6,https://quotes.toscrape.com/js/page/6/,10,10,60,0.086,2026-11-17T11:29:38+00:00,ok
6,7,js,7,https://quotes.toscrape.com/js/page/7/,10,10,70,0.082,2026-11-17T11:29:39+00:00,ok
7,8,js,8,https://quotes.toscrape.com/js/page/8/,10,10,80,0.079,2026-11-17T11:29:41+00:00,ok
8,9,js,9,https://quotes.toscrape.com/js/page/9/,10,10,90,0.090,2026-11-17T11:29:42+00:00,ok
9,10,js,10,https://quotes.toscrape.com/js/page/10/,10,10,100,0.087,2026-11-17T11:29:44+00:00,ok


## 4. Диагностируемость сбоев

Контролируемый эксперимент на `/js-delayed/`: таймаут ожидания отрисовки намеренно ставится 3 с, а площадка задерживает отрисовку дольше. Что должно произойти:
- на каждой странице перехватывается `TimeoutError`;
- сохраняются скриншот и HTML;
- в журнал пишется `status=timeout`;
- сбор **продолжается** со следующей страницы (переход по Next тоже работает, пагинатор отдаётся сервером);
- срабатывает предохранитель «нет прироста N раз подряд».

In [14]:
async def failure_demo(browser):
    ctx = await new_context(browser, block_media=True, storage_state=STATE_FILE)
    try:
        return await collect_source(ctx, "js-delayed", item_timeout=3_000, max_pages=3)
    finally:
        await ctx.close()

fail_recs, fail_log, fail_reason = await with_browser(failure_demo)
df_fail_log = pd.DataFrame(fail_log)
df_fail_log.to_csv(ART / "failure_demo_log.csv", index=False, encoding="utf-8-sig")
print("Записей:", len(fail_recs), "| причина остановки:", fail_reason)
display(df_fail_log)
print("Артефакты:", sorted(p.name for p in ART.glob("js-delayed_p*")))

  ⚠ [js-delayed] стр.1: Locator.wait_for: Timeout 3000ms exceeded. → артефакты js-delayed_p1_20261117_143145.*
  ⚠ [js-delayed] стр.2: Locator.wait_for: Timeout 3000ms exceeded. → артефакты js-delayed_p2_20261117_143150.*
Записей: 0 | причина остановки: stall


,source,iteration,url,elements,increment,total,duration_s,timestamp,status
0,js-delayed,1,https://quotes.toscrape.com/js-delayed/,0,0,0,3.130,2026-11-17T11:31:45+00:00,timeout
1,js-delayed,2,https://quotes.toscrape.com/js-delayed/page/2/,0,0,0,3.106,2026-11-17T11:31:50+00:00,timeout


Артефакты: ['js-delayed_p1_20261117_124205.html', 'js-delayed_p1_20261117_124205.png', 'js-delayed_p1_20261117_143145.html', 'js-delayed_p1_20261117_143145.png', 'js-delayed_p2_20261117_124209.html', 'js-delayed_p2_20261117_124209.png', 'js-delayed_p2_20261117_143150.html', 'js-delayed_p2_20261117_143150.png']


## 5. Дешёвая альтернатива — вариант A: JSON из `<script>`

Диагностика показала, что данные лежат в исходном HTML в `var data = [...]`. Поэтому браузер не нужен: достаточно `requests` и `json.loads`.

- Пагинация идёт по адресам `/page/N/`.
- Предохранители: максимум страниц, пустая порция, лимит записей, отсутствие ссылки на следующую страницу.
- Между запросами выдерживается пауза вежливости, при сбоях работают повторы через `Retry`.
- Сверка с браузерным набором выполняется по ключу `(source, text)`; сравниваются автор, теги и позиция.

In [15]:
def collect_light(source, *, max_pages=MAX_PAGES, max_items=PER_SOURCE_LIMIT, pause=POLITE_PAUSE):
    out, log, stats = [], [], {"requests": 0, "bytes": 0}
    stop_reason = "max_pages"
    for page_no in range(1, max_pages + 1):
        url = SOURCES[source] if page_no == 1 else f"{SOURCES[source]}page/{page_no}/"
        t0 = time.perf_counter()
        r = SESSION.get(url, timeout=20)
        stats["requests"] += 1
        stats["bytes"] += int(r.headers.get("Content-Length", len(r.content))) + sum(len(k) + len(v) for k, v in r.headers.items())
        r.raise_for_status()
        m = DATA_RE.search(r.text)
        if not m:
            raise ValueError(f"На {url} не найден блок var data = [...] — проверьте протокол диагностики")
        batch = json.loads(m.group(1))
        for i, d in enumerate(batch, 1):
            a = d.get("author") or {}
            out.append({"source": source, "page": page_no, "position": i,
                        "text": d.get("text"), "author": a.get("name"),
                        "tags": "; ".join(d.get("tags") or []),
                        "author_slug": a.get("slug"), "author_goodreads_link": a.get("goodreads_link"),
                        "page_url": url, "collected_at": now_iso()})
        log.append({"source": source, "iteration": page_no, "url": url, "elements": len(batch),
                    "increment": len(batch), "total": len(out), "status_code": r.status_code,
                    "duration_s": round(time.perf_counter() - t0, 3), "timestamp": now_iso()})
        if not batch:
            stop_reason = "empty_page"; break
        if len(out) >= max_items:
            stop_reason = "max_items"; break
        if f"/page/{page_no + 1}/" not in r.text:
            stop_reason = "last_page"; break
        time.sleep(pause)                                          # пауза вежливости
    return out[:max_items], log, stats, stop_reason

with PeakMem() as mem_light:
    t0 = time.perf_counter()
    light_recs, light_log, light_stats = [], [], {"requests": 0, "bytes": 0}
    for src in SOURCES:
        recs, log, stats, reason = collect_light(src)
        light_recs += recs; light_log += log
        light_stats = {k: light_stats[k] + stats[k] for k in light_stats}
        print(f"[{src}] записей: {len(recs)}, запросов: {stats['requests']}, остановка: {reason}")
    light_time = time.perf_counter() - t0

df_light = pd.DataFrame(light_recs)
df_light.to_csv("data_light.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(light_log).to_csv("collection_log_light.csv", index=False, encoding="utf-8-sig")
print(f"\nИтого: {len(df_light)} записей за {light_time:.1f} c | запросов {light_stats['requests']}, "
      f"трафик {light_stats['bytes']/1e6:.3f} МБ | пиковая память {mem_light.peak_mb} МБ")
print("Поля, которых нет в DOM, но есть в JSON:", ["author_slug", "author_goodreads_link"])

[js] записей: 100, запросов: 10, остановка: max_items
[js-delayed] записей: 100, запросов: 10, остановка: max_items

Итого: 200 записей за 13.7 c | запросов 20, трафик 0.127 МБ | пиковая память 155.4 МБ
Поля, которых нет в DOM, но есть в JSON: ['author_slug', 'author_goodreads_link']


In [16]:
# Сверка наборов: data_light.csv vs data_browser.csv
def norm(s):
    return re.sub(r"\s+", " ", str(s)).strip() if isinstance(s, str) else ""

b = df_browser.assign(k_text=df_browser["text"].map(norm))
l = df_light.assign(k_text=df_light["text"].map(norm))
m = b.merge(l, on=["source", "k_text"], how="outer", suffixes=("_browser", "_light"), indicator=True)

both    = m[m["_merge"] == "both"]
only_b  = m[m["_merge"] == "left_only"]
only_l  = m[m["_merge"] == "right_only"]
auth_mm = both[both["author_browser"].map(norm) != both["author_light"].map(norm)]
tags_mm = both[both["tags_browser"].fillna("") != both["tags_light"].fillna("")]
pos_mm  = both[(both["page_browser"] != both["page_light"]) | (both["position_browser"] != both["position_light"])]

summary = pd.DataFrame([
    {"проверка": "записей в браузерном наборе",         "значение": len(df_browser)},
    {"проверка": "записей в лёгком наборе",             "значение": len(df_light)},
    {"проверка": "совпало по ключу (source, text)",     "значение": len(both)},
    {"проверка": "только в браузерном",                 "значение": len(only_b)},
    {"проверка": "только в лёгком",                     "значение": len(only_l)},
    {"проверка": "расхождение автора",                  "значение": len(auth_mm)},
    {"проверка": "расхождение тегов",                   "значение": len(tags_mm)},
    {"проверка": "расхождение страницы/позиции",        "значение": len(pos_mm)},
])
display(summary)

cols = ["source", "k_text", "author_browser", "author_light", "tags_browser", "tags_light",
        "page_browser", "page_light", "position_browser", "position_light"]
discrepancies = pd.concat([
    only_b.assign(тип="только в браузерном"), only_l.assign(тип="только в лёгком"),
    auth_mm.assign(тип="автор"), tags_mm.assign(тип="теги"), pos_mm.assign(тип="страница/позиция"),
])[["тип"] + cols]
discrepancies.to_csv("discrepancies.csv", index=False, encoding="utf-8-sig")
summary.to_csv("reconciliation_summary.csv", index=False, encoding="utf-8-sig")
print("Расхождений всего:", len(discrepancies), "→ discrepancies.csv")
if len(discrepancies):
    display(discrepancies.head(20))

,проверка,значение
0,записей в браузерном наборе,200
1,записей в лёгком наборе,200
2,"совпало по ключу (source, text)",200
3,только в браузерном,0
4,только в лёгком,0
5,расхождение автора,0
6,расхождение тегов,0
7,расхождение страницы/позиции,0


Расхождений всего: 0 → discrepancies.csv


## 6. Замеры и оптимизация

**6.1 Режимы браузера.** Один и тот же сбор `/js/` (10 страниц) выполняется в нескольких режимах, каждый в отдельном браузере, чтобы честно замерить память.

**Важная тонкость.** Если включён перехват `page.route`, Playwright **отключает HTTP-кэш браузера**. Без перехвата стили и шрифты скачиваются один раз, а на страницах 2–10 берутся из кэша. С перехватом они скачиваются заново на каждой странице. Поэтому сравнивать «блокировка» с «без перехвата вообще» нечестно: блокировка выглядит хуже, хотя дело в кэше. Базой служит режим с перехватом, который ничего не блокирует. Режим без перехвата показан отдельно, для справки.

| Режим | Перехват | Блокировка `image/media/font` | Ожидание загрузки |
|---|---|---|---|
| **A0** (справочно) | нет, кэш работает | нет | `load` |
| **A** — база | есть, пропускает всё | нет | `load` |
| **B** | есть | да | `domcontentloaded` |
| **C** (вариант C задания) | есть | да | `commit` |

Каждый режим выполняется 3 раза, берётся медиана времени. Время показано и полное, и **без пауз вежливости**: паузы одинаковы во всех режимах и только разбавляют разницу.

**6.2** Сводная таблица метрик. Измеренные столбцы заполняет код. Время разработки, устойчивость, воспроизводимость и столбец No-code вы вписываете сами после прогона n8n.

In [17]:
async def _pass_router(route):
    await route.continue_()                                        # перехват без блокировки (кэш отключается так же)

async def measure_mode(label, *, route, block, wait_until, repeats=3):
    async def job(browser):
        ctx = await new_context(browser, block_media=block)
        if route and not block:
            await ctx.route("**/*", _pass_router)
        net = NetStats().attach(ctx)
        try:
            t0 = time.perf_counter()
            recs, log, _ = await collect_source(ctx, "js", wait_until=wait_until)
            elapsed = time.perf_counter() - t0
            await net.flush()
        finally:
            await ctx.close()
        return recs, elapsed, net, len(log)
    times, net_times, peak = [], [], 0
    for _ in range(repeats):
        with PeakMem() as mem:
            recs, elapsed, net, iters = await with_browser(job)
        times.append(elapsed)
        net_times.append(elapsed - POLITE_PAUSE * (iters - 1))     # минус паузы вежливости
        peak = max(peak, mem.peak_mb)
    return {"режим": label, "записей": len(recs),
            "время, с (медиана)": round(statistics.median(times), 2),
            "время без пауз, с": round(statistics.median(net_times), 2),
            "запросов": net.requests, "заблокировано": net.failed,
            "трафик, КБ": round(net.bytes / 1024, 1), "пиковая память, МБ": peak,
            "типы": dict(net.types)}

modes = [
    await measure_mode("A0: без перехвата, кэш работает (справочно)", route=False, block=False, wait_until="load"),
    await measure_mode("A: перехват без блокировки, load (база)",      route=True,  block=False, wait_until="load"),
    await measure_mode("B: блокировка медиа, domcontentloaded",        route=True,  block=True,  wait_until="domcontentloaded"),
    await measure_mode("C: блокировка медиа, commit",                  route=True,  block=True,  wait_until="commit"),
]
df_modes = pd.DataFrame(modes)
base = df_modes.iloc[1]                                            # база — режим A
df_modes["экономия времени без пауз, %"] = (100 * (1 - df_modes["время без пауз, с"] / base["время без пауз, с"])).round(1)
df_modes["экономия трафика, %"]          = (100 * (1 - df_modes["трафик, КБ"] / max(base["трафик, КБ"], 1e-9))).round(1)
df_modes.to_csv("blocking_effect.csv", index=False, encoding="utf-8-sig")
display(df_modes)

,режим,записей,"время, с (медиана)","время без пауз, с",запросов,заблокировано,"трафик, КБ","пиковая память, МБ",типы,"экономия времени без пауз, %","экономия трафика, %"
0,"A0: без перехвата, кэш работает (справочно)",100,12.28,7.78,70,0,157.8,546.2,"{'document': 10, 'stylesheet': 30, 'script': 10, 'font': 20}",43.1,86.2
1,"A: перехват без блокировки, load (база)",100,18.18,13.68,70,0,1146.1,559.0,"{'document': 10, 'stylesheet': 30, 'script': 10, 'font': 20}",0.0,0.0
2,"B: блокировка медиа, domcontentloaded",100,15.95,11.45,70,20,567.8,554.0,"{'document': 10, 'stylesheet': 30, 'script': 10, 'font': 20}",16.3,50.5
3,"C: блокировка медиа, commit",100,18.38,13.88,70,20,567.9,546.7,"{'document': 10, 'stylesheet': 30, 'script': 10, 'font': 20}",-1.5,50.4


In [18]:
# Сводная таблица (измеряемые строки заполнены, остальные — вручную)
nan = float("nan")
metrics = pd.DataFrame({
    "Playwright": {
        "Время разработки, мин":             nan,
        "Записей собрано":                   len(df_browser),
        "Время одного прогона, с":           round(browser_time, 2),
        "Число сетевых запросов":            net_b.requests,
        "Трафик, МБ":                        round(net_b.bytes / 1e6, 3),
        "Пиковая память процесса, МБ":       mem_browser.peak_mb,
        "Секунд на запись":                  round(browser_time / len(df_browser), 4),
        "Устойчивость к редизайну (1–5)":    nan,
        "Воспроизводимость (Git / экспорт)": "Git: lab4.ipynb, collect.py",
    },
    "Дешёвая альтернатива (JSON в <script>)": {
        "Время разработки, мин":             nan,
        "Записей собрано":                   len(df_light),
        "Время одного прогона, с":           round(light_time, 2),
        "Число сетевых запросов":            light_stats["requests"],
        "Трафик, МБ":                        round(light_stats["bytes"] / 1e6, 3),
        "Пиковая память процесса, МБ":       mem_light.peak_mb,
        "Секунд на запись":                  round(light_time / len(df_light), 4),
        "Устойчивость к редизайну (1–5)":    nan,
        "Воспроизводимость (Git / экспорт)": "Git: lab4.ipynb",
    },
    "No-code (Octoparse)": {
        "Время разработки, мин":             20,
        "Записей собрано":                   100,  # 110 изначально, но 10 дублей удалено
        "Время одного прогона, с":           40,   # Примерное время работы в Octoparse
        "Число сетевых запросов":            50,   # Браузер грузит страницу + AJAX)
        "Трафик, МБ":                        2.0,
        "Пиковая память процесса, МБ":       600,  # Chromium Octoparse
        "Секунд на запись":                  0.4,  # 40 / 100
        "Устойчивость к редизайну (1–5)":    2,    # Селекторы Octoparse ломаются при изменении структуры
        "Воспроизводимость (Git / экспорт)": "Экспорт JSON шаблона",
    },
})
metrics.to_csv("metrics_comparison.csv", encoding="utf-8-sig")
display(metrics)
print("Массовое vs поэлементное извлечение (раздел 2):", extract_cmp)
print("Примечания к замерам:",
      "\n • время Playwright включает обе площадки, в т.ч. задержку отрисовки /js-delayed/ и паузы вежливости;",
      "\n • число запросов браузера включает заблокированные (они отменяются до отправки);",
      "\n • память браузера = Python + драйвер + Chromium; лёгкого способа = только Python;",
      "\n • No-code метрики даны для Octoparse (110 строк собрано, 100 уникальных после удаления дублей);",
      "\n • трафик и число запросов в Octoparse измерены приблизительно.")

,Playwright,Дешёвая альтернатива (JSON в <script>),No-code (Octoparse)
"Время разработки, мин",NaN,NaN,20
Записей собрано,200,200,100
"Время одного прогона, с",133.01,13.74,40
Число сетевых запросов,146,20,50
"Трафик, МБ",1.189,0.127,2.0
"Пиковая память процесса, МБ",560.2,155.4,600
Секунд на запись,0.665,0.0687,0.4
Устойчивость к редизайну (1–5),NaN,NaN,2
Воспроизводимость (Git / экспорт),"Git: lab4.ipynb, collect.py",Git: lab4.ipynb,Экспорт JSON шаблона


Массовое vs поэлементное извлечение (раздел 2): {'evaluate_all, мс (медиана)': 6.5, 'поэлементно, мс (медиана)': 136.6, 'выигрыш, раз': 20.9}
Примечания к замерам: 
 • время Playwright включает обе площадки, в т.ч. задержку отрисовки /js-delayed/ и паузы вежливости; 
 • число запросов браузера включает заблокированные (они отменяются до отправки); 
 • память браузера = Python + драйвер + Chromium; лёгкого способа = только Python; 
 • No-code метрики даны для Octoparse (110 строк собрано, 100 уникальных после удаления дублей); 
 • трафик и число запросов в Octoparse измерены приблизительно.


## 7. Качество данных и smoke-check

- **Проверка качества:** число записей, доля пустых обязательных полей, дубликаты по ключу `(source, text)`, приведение типов.
- **Smoke-check:** функция печатает предупреждение, если доля пустых значений ключевого поля выше порога (5 %). Ниже показана работа на реальном наборе и на искусственно испорченной копии, где срабатывание ожидаемо.

In [19]:
REQUIRED = ("text", "author")

def typed(df):
    out = df.copy()
    out["page"]         = pd.to_numeric(out["page"], errors="coerce").astype("Int64")
    out["position"]     = pd.to_numeric(out["position"], errors="coerce").astype("Int64")
    out["collected_at"] = pd.to_datetime(out["collected_at"], errors="coerce", utc=True)
    out["tags"]         = out["tags"].fillna("").astype("string")
    out["tags_count"]   = out["tags"].map(lambda s: len([t for t in s.split(";") if t.strip()])).astype("Int64")
    for c in ("source", "text", "author", "page_url"):
        out[c] = out[c].astype("string")
    return out

def empty_mask(s):
    return s.isna() | (s.astype("string").str.strip() == "")

def quality_report(df, name, key=("source", "text"), required=REQUIRED):
    rep = {"набор": name, "записей": len(df)}
    for c in required:
        rep[f"пустых «{c}», %"] = round(100 * empty_mask(df[c]).mean(), 2)
    rep["без тегов, %"]          = round(100 * empty_mask(df["tags"]).mean(), 2)
    rep["дубликатов по ключу"]   = int(df.duplicated(subset=list(key)).sum())
    rep["по источникам"]         = df["source"].value_counts().to_dict()
    rep["нераспознанных типов"]  = int(df["page"].isna().sum() + df["collected_at"].isna().sum())
    return rep

def smoke_check(df, fields=("text", "author", "tags"), threshold=0.05, name=""):
    ok = True
    for f in fields:
        if f not in df.columns:
            print(f"⚠ [{name}] поле «{f}» отсутствует"); ok = False; continue
        share = empty_mask(df[f]).mean()
        bad = share > threshold
        print(f"{'⚠ ПРЕДУПРЕЖДЕНИЕ' if bad else '✓'} [{name}] «{f}»: пустых {share:.1%} (порог {threshold:.0%})")
        ok = ok and not bad
    return ok

tb, tl = typed(df_browser), typed(df_light)
print(tb.dtypes, "\n")
quality = pd.DataFrame([quality_report(tb, "data_browser"), quality_report(tl, "data_light")])
quality.to_csv("quality_report.csv", index=False, encoding="utf-8-sig")
display(quality)

print("\nSmoke-check реальных наборов:")
smoke_ok_b = smoke_check(tb, name="data_browser")
smoke_ok_l = smoke_check(tl, name="data_light")

print("\nSmoke-check испорченной копии (у 10% записей стёрт автор) — должно сработать:")
broken = tb.copy()
broken.loc[broken.sample(frac=0.10, random_state=1).index, "author"] = ""
smoke_ok_broken = smoke_check(broken, name="broken_demo")
print("Результат smoke-check испорченной копии:", smoke_ok_broken)

source               string[python]
page                          Int64
position                      Int64
text                 string[python]
author               string[python]
tags                 string[python]
page_url             string[python]
collected_at    datetime64[ns, UTC]
tags_count                    Int64
dtype: object 



,набор,записей,"пустых «text», %","пустых «author», %","без тегов, %",дубликатов по ключу,по источникам,нераспознанных типов
0,data_browser,200,0.0,0.0,3.0,0,"{'js': 100, 'js-delayed': 100}",0
1,data_light,200,0.0,0.0,3.0,0,"{'js': 100, 'js-delayed': 100}",0



Smoke-check реальных наборов:
✓ [data_browser] «text»: пустых 0.0% (порог 5%)
✓ [data_browser] «author»: пустых 0.0% (порог 5%)
✓ [data_browser] «tags»: пустых 3.0% (порог 5%)
✓ [data_light] «text»: пустых 0.0% (порог 5%)
✓ [data_light] «author»: пустых 0.0% (порог 5%)
✓ [data_light] «tags»: пустых 3.0% (порог 5%)

Smoke-check испорченной копии (у 10% записей стёрт автор) — должно сработать:
✓ [broken_demo] «text»: пустых 0.0% (порог 5%)
⚠ ПРЕДУПРЕЖДЕНИЕ [broken_demo] «author»: пустых 10.0% (порог 5%)
✓ [broken_demo] «tags»: пустых 3.0% (порог 5%)
Результат smoke-check испорченной копии: False


## 8. Локаторы и устойчивость к редизайну

**8.1** Таблица локаторов, использованных в коде, с уровнем по таблице приоритетов занятия. В отрендеренном DOM `/js/` нет `data-*`- и `itemprop`-атрибутов (замер в разделе 2), поэтому поля цитаты берутся по классам компонента (уровень 5). Структурного XPath и сгенерированных классов нет.

**8.2** Стенд редизайна. Через `page.set_content` создаются варианты разметки:
- исходная;
- с переименованными классами;
- с дополнительной `div`-обёрткой;
- с кнопкой на русском языке.

На каждом варианте запускаются использованные локаторы и альтернативы к ним. На выходе — матрица «сколько элементов найдено». На каждом варианте ожидается 3 цитаты и 1 кнопка.

In [20]:
locators_used = pd.DataFrame([
    {"локатор": 'get_by_role("link", name="Next")',     "где": "пагинация, клик",            "тип": "роль + доступное имя", "приоритет": 2},
    {"локатор": 'get_by_role("button", name="Login")',  "где": "форма входа",                "тип": "роль + доступное имя", "приоритет": 2},
    {"локатор": 'get_by_role("link", name="Logout")',   "где": "проверка авторизации",       "тип": "роль + доступное имя", "приоритет": 2},
    {"локатор": 'get_by_label("Username"/"Password")',  "где": "форма входа",                "тип": "метка поля",           "приоритет": 3},
    {"локатор": '.quote',                               "где": "контейнер цитаты, ожидание", "тип": "класс компонента",     "приоритет": 5},
    {"локатор": '.text / .author / .tag (внутри .quote)', "где": "поля в evaluate_all",      "тип": "класс компонента",     "приоритет": 5},
])
locators_used["data-/itemprop-атрибуты в DOM /js/"] = f"data-*: {dom_attrs['data_attr']}, itemprop: {dom_attrs['itemprop']}"
locators_used["обоснование устойчивости"] = "— заполнить в отчёте —"
locators_used.to_csv("locators_table.csv", index=False, encoding="utf-8-sig")
display(locators_used)

,локатор,где,тип,приоритет,data-/itemprop-атрибуты в DOM /js/,обоснование устойчивости
0,"get_by_role(""link"", name=""Next"")","пагинация, клик",роль + доступное имя,2,"data-*: 0, itemprop: 0",— заполнить в отчёте —
1,"get_by_role(""button"", name=""Login"")",форма входа,роль + доступное имя,2,"data-*: 0, itemprop: 0",— заполнить в отчёте —
2,"get_by_role(""link"", name=""Logout"")",проверка авторизации,роль + доступное имя,2,"data-*: 0, itemprop: 0",— заполнить в отчёте —
3,"get_by_label(""Username""/""Password"")",форма входа,метка поля,3,"data-*: 0, itemprop: 0",— заполнить в отчёте —
4,.quote,"контейнер цитаты, ожидание",класс компонента,5,"data-*: 0, itemprop: 0",— заполнить в отчёте —
5,.text / .author / .tag (внутри .quote),поля в evaluate_all,класс компонента,5,"data-*: 0, itemprop: 0",— заполнить в отчёте —


In [21]:
def stand_html(qcls="quote", tcls="text", acls="author", wrap=False, next_text="Next", itemprop=False):
    ip = (lambda v: f' itemprop="{v}"') if itemprop else (lambda v: "")
    items = []
    for i in range(1, 4):
        inner = f'<span class="{tcls}"{ip("text")}>“Цитата {i}”</span>'
        if wrap:
            inner = f'<div class="wrapper">{inner}</div>'
        items.append(f'<div class="{qcls}">{inner}<span>by <small class="{acls}"{ip("author")}>Автор {i}</small></span>'
                     f'<div class="tags">Tags: <a class="tag">t{i}</a></div></div>')
    body = "".join(items)
    if wrap:
        body = f'<div class="container-new">{body}</div>'
    return (f'<html><body>{body}<nav><ul class="pager"><li class="next">'
            f'<a href="#">{next_text} <span aria-hidden="true">&rarr;</span></a></li></ul></nav></body></html>')

VARIANTS = {
    "исходная (как /js/)":            stand_html(),
    "переименованы классы":           stand_html(qcls="q-card", tcls="q-body", acls="q-by", itemprop=True),
    "доп. div-обёртка":               stand_html(wrap=True),
    "кнопка на русском («Далее»)":    stand_html(next_text="Далее"),
}
CANDIDATES = {
    "L1  .quote .text (использован)":            lambda p: p.locator(".quote .text"),
    "L1a .quote > .text (строгий дочерний)":     lambda p: p.locator(".quote > .text"),
    "L1b [itemprop=text] (атрибут данных)":      lambda p: p.locator("[itemprop=text]"),
    "L2  role=link 'Next' (использован)":        lambda p: p.get_by_role("link", name="Next"),
    "L2a role=link /Next|Далее/ (регэксп имени)": lambda p: p.get_by_role("link", name=re.compile(r"Next|Далее", re.I)),
    "L2b .pager .next → role=link (без текста)":  lambda p: p.locator(".pager .next").get_by_role("link"),
    "L3  .quote .author (использован)":          lambda p: p.locator(".quote .author"),
    "L3a [itemprop=author] (атрибут данных)":    lambda p: p.locator("[itemprop=author]"),
}

async def resilience_stand(browser):
    ctx = await new_context(browser)
    try:
        page = await ctx.new_page()
        rows = {}
        for vname, html in VARIANTS.items():
            await page.set_content(html)
            rows[vname] = {cname: await make(page).count() for cname, make in CANDIDATES.items()}
        return rows
    finally:
        await ctx.close()

resilience = pd.DataFrame(await with_browser(resilience_stand))
resilience.to_csv("resilience_matrix.csv", encoding="utf-8-sig")
print("Число найденных элементов (ожидается 3 для полей и 1 для кнопки):")
display(resilience)

Число найденных элементов (ожидается 3 для полей и 1 для кнопки):


,исходная (как /js/),переименованы классы,доп. div-обёртка,кнопка на русском («Далее»)
L1 .quote .text (использован),3,0,3,3
L1a .quote > .text (строгий дочерний),3,0,0,3
L1b [itemprop=text] (атрибут данных),0,3,0,0
L2 role=link 'Next' (использован),1,1,1,0
L2a role=link /Next|Далее/ (регэксп имени),1,1,1,1
L2b .pager .next → role=link (без текста),1,1,1,1
L3 .quote .author (использован),3,0,3,3
L3a [itemprop=author] (атрибут данных),0,3,0,0


## 9. CLI-скрипт с кодами возврата

`collect.py` использует sync API. Его можно запустить локально на Windows: `python collect.py --headed`.

Коды возврата:

| Код | Значение |
|---|---|
| `0` | успех |
| `1` | непредвиденная ошибка |
| `2` | собрано меньше `--min-items` |
| `3` | smoke-check не пройден |
| `4` | были таймауты страниц (только при `--strict`) |

Ниже два запуска: обычный (ожидается код 0) и с заведомо невыполнимым `--min-items` (ожидается код 2).

In [22]:
%%writefile collect.py
#!/usr/bin/env python
"""Лаб. 4, вариант 2: браузерный сбор quotes.toscrape.com/js/ и /js-delayed/ (sync API).

Коды возврата: 0 — успех; 1 — непредвиденная ошибка; 2 — собрано меньше --min-items;
3 — smoke-check не пройден; 4 — были таймауты страниц (только с --strict).
"""
import argparse, csv, re, sys, time
from datetime import datetime, timezone
from pathlib import Path
from playwright.sync_api import sync_playwright, expect, TimeoutError as PWTimeout

BASE = "https://quotes.toscrape.com"
SOURCES = {"js": f"{BASE}/js/", "js-delayed": f"{BASE}/js-delayed/"}
UA = "course-crawler/1.0 (educational use; lab4 variant 2)"
ITEM_TIMEOUT = {"js": 10_000, "js-delayed": 25_000}
EXTRACT_JS = """els => els.map((el, i) => ({position: i + 1,
    text: el.querySelector('.text')?.innerText?.trim() ?? null,
    author: el.querySelector('.author')?.innerText?.trim() ?? null,
    tags: Array.from(el.querySelectorAll('.tag')).map(t => t.innerText.trim()).join('; ')}))"""

def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def page_number(url):
    m = re.search(r"/page/(\d+)/", url)
    return int(m.group(1)) if m else 1

def block(route):
    if route.request.resource_type in {"image", "media", "font"}:
        route.abort()
    else:
        route.continue_()

def ensure_auth(browser, state):
    if state.exists():
        return
    ctx = browser.new_context(user_agent=UA)
    try:
        page = ctx.new_page()
        page.goto(f"{BASE}/login")
        page.get_by_label("Username").fill("student")
        page.get_by_label("Password").fill("student")
        page.get_by_role("button", name="Login").click()
        expect(page.get_by_role("link", name="Logout")).to_be_visible()
        ctx.storage_state(path=str(state))
    finally:
        ctx.close()

def collect(ctx, source, art, max_pages, max_items, stall_limit, pause):
    page = ctx.new_page()
    recs, log, seen, stalls, timeouts = [], [], set(), 0, 0
    try:
        page.goto(SOURCES[source], wait_until="domcontentloaded")
        for it in range(1, max_pages + 1):
            t0, url, pno, status = time.perf_counter(), page.url, page_number(page.url), "ok"
            try:
                page.locator(".quote").first.wait_for(state="visible", timeout=ITEM_TIMEOUT[source])
                batch = page.locator(".quote").evaluate_all(EXTRACT_JS)
            except PWTimeout:
                status, batch, timeouts = "timeout", [], timeouts + 1
                base = art / f"{source}_p{pno}_{datetime.now():%Y%m%d_%H%M%S}"
                page.screenshot(path=f"{base}.png", full_page=True)
                Path(f"{base}.html").write_text(page.content(), encoding="utf-8")
                print(f"[WARN] {source} стр.{pno}: таймаут, артефакты {base.name}.*", file=sys.stderr)
            new = 0
            for r in batch:
                if r["text"] and (source, r["text"]) in seen:
                    continue
                seen.add((source, r["text"]))
                r.update(source=source, page=pno, page_url=url, collected_at=now_iso())
                recs.append(r); new += 1
            log.append({"source": source, "iteration": it, "url": url, "elements": len(batch),
                        "increment": new, "total": len(recs),
                        "duration_s": round(time.perf_counter() - t0, 3), "timestamp": now_iso(),
                        "status": status})
            stalls = stalls + 1 if new == 0 else 0
            if stalls >= stall_limit or len(recs) >= max_items:
                break
            nxt = page.get_by_role("link", name="Next")
            if nxt.count() == 0:
                break
            time.sleep(pause)                       # пауза вежливости
            nxt.click()
            page.wait_for_url(f"**/page/{pno + 1}/", wait_until="domcontentloaded")
    finally:
        page.close()
    return recs[:max_items], log, timeouts

def write_csv(path, rows):
    if not rows:
        path.write_text("", encoding="utf-8-sig"); return
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)

def main(argv=None):
    for stream in (sys.stdout, sys.stderr):        # кириллица в консоли Windows (cp866/cp1251)
        if hasattr(stream, "reconfigure"):
            stream.reconfigure(encoding="utf-8", errors="replace")
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--source", choices=["js", "js-delayed", "all"], default="all")
    ap.add_argument(
    "--out-dir",
    type=Path,
    default=".",
    help="каталог для сохранения результатов"
    ap.add_argument("--max-items", type=int, default=100, help="лимит записей на источник")
    ap.add_argument("--max-pages", type=int, default=15)
    ap.add_argument("--stall-limit", type=int, default=2)
    ap.add_argument("--min-items", type=int, default=200, help="минимум записей для кода 0")
    ap.add_argument("--pause", type=float, default=0.5)
    ap.add_argument("--headed", action="store_true", help="показать окно браузера")
    ap.add_argument("--strict", action="store_true", help="код 4 при любых таймаутах")
    a = ap.parse_args(argv)

    out = Path(a.out_dir); art = out / "artifacts"; art.mkdir(parents=True, exist_ok=True)
    state = out / "auth_state.json"
    sources = list(SOURCES) if a.source == "all" else [a.source]
    recs, log, timeouts = [], [], 0
    try:
        with sync_playwright() as pw:
            browser = pw.chromium.launch(headless=not a.headed)
            try:
                ensure_auth(browser, state)
                ctx = browser.new_context(user_agent=UA, viewport={"width": 1280, "height": 900},
                                          locale="ru-RU", storage_state=str(state))
                ctx.set_default_timeout(15_000)
                ctx.set_default_navigation_timeout(30_000)
                ctx.route("**/*", block)
                try:
                    for s in sources:
                        r, l, t = collect(ctx, s, art, a.max_pages, a.max_items, a.stall_limit, a.pause)
                        recs += r; log += l; timeouts += t
                finally:
                    ctx.close()
            finally:
                browser.close()
    except Exception as e:
        print(f"[FATAL] {type(e).__name__}: {e}", file=sys.stderr)
        return 1

    write_csv(out / "data_browser.csv", recs)
    write_csv(out / "collection_log.csv", log)
    print(f"[INFO] записей: {len(recs)}, итераций: {len(log)}, таймаутов: {timeouts} → {out}/")

    for field in ("text", "author"):
        share = sum(1 for r in recs if not (r.get(field) or "").strip()) / max(len(recs), 1)
        if share > 0.05:
            print(f"[ERROR] smoke-check: пустых «{field}» {share:.1%} > 5%", file=sys.stderr)
            return 3
    if len(recs) < a.min_items:
        print(f"[ERROR] собрано {len(recs)} < --min-items {a.min_items}", file=sys.stderr)
        return 2
    if a.strict and timeouts:
        return 4
    return 0

if __name__ == "__main__":
    sys.exit(main())

Overwriting collect.py


In [26]:
import os
import sys
import subprocess

def run_cli(*args):
    """Запуск collect.py отдельным процессом и вывод кода возврата."""
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}

    r = subprocess.run(
        [sys.executable, "collect.py", *args],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env
    )

    if r.stdout.strip():
        print(r.stdout)
    if r.stderr.strip():
        print(r.stderr)

    print("exit code:", r.returncode)
    return r.returncode

rc_ok = run_cli("--source", "js", "--max-items", "30", "--min-items", "30")

[INFO] записей: 30, итераций: 3, таймаутов: 0 → ./

exit code: 0


In [28]:
rc_fail = run_cli("--source", "js", "--max-items", "20", "--min-items", "500")   # ожидается 2
rc_all = run_cli(
    "--source", "all",
    "--max-items", "100",
    "--min-items", "200"
)

[INFO] записей: 20, итераций: 2, таймаутов: 0 → ./

[ERROR] собрано 20 < --min-items 500

exit code: 2
[INFO] записей: 200, итераций: 20, таймаутов: 0 → ./

exit code: 0


## 10. Завершение: проверка утечек и состав результатов

Все браузеры открывались через `with_browser` и закрыты в `finally`. Ниже это проверяется: дочерних процессов Chromium остаться не должно.

In [25]:
def leftover_browsers():
    found = []
    for p in psutil.Process().children(recursive=True):
        try:
            name = p.name().lower()
        except psutil.Error:
            continue
        if "chrom" in name or "headless" in name:
            found.append(f"{p.pid}:{name}")
    return found

left = leftover_browsers()
print("Оставшиеся процессы браузера:", left or "нет ✓")

files = ["data_browser.csv", "data_light.csv", "collection_log.csv", "collection_log_light.csv",
         "diagnostics_protocol.csv", "reconciliation_summary.csv", "discrepancies.csv",
         "blocking_effect.csv", "metrics_comparison.csv", "quality_report.csv",
         "locators_table.csv", "resilience_matrix.csv", "auth_state.json", "collect.py"]
display(pd.DataFrame([{"файл": f, "есть": Path(f).exists(),
                       "размер, байт": Path(f).stat().st_size if Path(f).exists() else 0} for f in files]))
print("artifacts/:", sorted(p.name for p in ART.iterdir()))

Оставшиеся процессы браузера: нет ✓


,файл,есть,"размер, байт"
0,data_browser.csv,True,49196
1,data_light.csv,True,58596
2,collection_log.csv,True,2073
3,collection_log_light.csv,True,2033
4,diagnostics_protocol.csv,True,1533
5,reconciliation_summary.csv,True,403
6,discrepancies.csv,True,133
7,blocking_effect.csv,True,970
8,metrics_comparison.csv,True,659
9,quality_report.csv,True,344


artifacts/: ['diag_js-delayed_js_off.png', 'diag_js_js_off.png', 'failure_demo_log.csv', 'js-delayed_p1_20261117_124205.html', 'js-delayed_p1_20261117_124205.png', 'js-delayed_p1_20261117_143145.html', 'js-delayed_p1_20261117_143145.png', 'js-delayed_p2_20261117_124209.html', 'js-delayed_p2_20261117_124209.png', 'js-delayed_p2_20261117_143150.html', 'js-delayed_p2_20261117_143150.png']
